# Gene Mapping Pipeline
Combined pipeline from the Step1 scripts; run the cells in order to produce the same intermediate files as the original pipeline.


## Requirements
Ensure the environment has `cobra`, `mygene`, `pandas`, and `scipy` installed; the cells assume the Step1 data files remain in the same folder structure.


In [ ]:
import re
import os
import mygene
import json
from cobra.io import load_json_model

class GeneMapper:
    def __init__(self, model_path, gene_file, common_genes_file ,gene_dict_file):
        self.model_path = model_path
        self.gene_file = gene_file
        self.gene_dict_file = gene_dict_file
        self.common_genes_file = common_genes_file
        self.gene_dict = {}
        self.gene_set = set()
        self.mg = mygene.MyGeneInfo()

    def load_genes_from_model(self):
        """Load gene IDs from a JSON model file."""
        model = load_json_model(self.model_path)
        gene_ids = set(gene.id for gene in model.genes)
        return gene_ids

    def write_gene_ids(self, gene_ids):
        """Write gene IDs to a text file."""
        with open(self.gene_file, 'w') as file:
            for gene in gene_ids:
                file.write(f'{gene}\n')

    def load_gene_ids(self):
        """Load gene IDs from a text file."""
        if os.path.exists(self.gene_file):
            with open(self.gene_file, 'r') as file:
                gene_ids = {line.strip() for line in file}
        else:
            gene_ids = self.load_genes_from_model()
            self.write_gene_ids(gene_ids)
        return gene_ids

    def convert_to_numeric_ids(self, gene_ids):
        """Extract numeric gene IDs from the loaded set."""
        return [re.match(r'(\d+)', gene_id).group(1) for gene_id in gene_ids if re.match(r'(\d+)', gene_id)]

    def map_names(self, genes):
        """Map Entrez gene IDs to gene symbols and update gene_dict and gene_set."""
        result = self.mg.querymany(genes, scopes="entrezgene", fields="symbol", species="human")
        for item in result:
            symbol = item.get('symbol', None)
            if symbol:
                self.gene_dict[item['query']] = [symbol]
                self.gene_set.add(symbol)
            else:
                print(f"Gene {item['query']} not found (no symbol available)")

    def process_multiple_ids(self, item, ensembl_id):
        """Handle multiple Ensembl IDs for a single gene."""
        for gene in ensembl_id:
            self.gene_set.add(gene['gene'])
            if item['query'] not in self.gene_dict:
                self.gene_dict[item['query']] = [gene['gene']]
            else:
                self.gene_dict[item['query']].append(gene['gene'])

    def save_gene_dict(self):
        """Save the gene dictionary to a JSON file."""
        with open(self.gene_dict_file, 'w') as file:
            json.dump(self.gene_dict, file)

    def save_gene_set(self):
        """Save the gene set to the gene file."""
        with open(self.common_genes_file, 'w') as file:
            for gene in self.gene_set:
                file.write(f'{gene}\n')

    def run_mapping(self):
        """Main function to run the mapping process."""
        gene_ids = self.load_gene_ids()
        numeric_gene_ids = self.convert_to_numeric_ids(gene_ids)
        self.map_names(numeric_gene_ids)
        self.save_gene_dict()
        self.save_gene_set()


In [ ]:
import csv
import re

# # Example usage
# csv_file_path = 'transcriptomics.csv'  # Replace with your actual CSV file path
# txt_file_path = 'ensembl_gene_ids.txt'  # Replace with your actual text file path
# gene_processor = GeneProcessor(csv_file_path, txt_file_path)

# # Find and save common genes
# gene_processor.save_common_genes()

class GeneProcessor:
    def __init__(self, csv_file, txt_file, output_file):
        self.csv_file = csv_file
        self.txt_file = txt_file
        self.output_file = output_file

    def get_values_in_first_line(self):
        """Extract gene values from the first line of the CSV file."""
        with open(self.csv_file, mode='r') as file:
            csv_reader = csv.reader(file)
            first_line = next(csv_reader)
            return first_line

    def get_values_from_lines(self):
        """Extract gene values from the text file, removing newline characters."""
        with open(self.txt_file, mode='r') as file:
            lines = file.readlines()
            return [line.strip() for line in lines]

    def process_gene_versions(self, genes):
        """Remove version numbers from gene IDs."""
        return [gene.split('.')[0] for gene in genes]

    def find_common_genes(self):
        """Find common genes between dataset genes and recon genes."""
        dataset_genes = self.get_values_in_first_line()
        recon_genes = self.get_values_from_lines()
        
        # Process dataset genes to ignore version numbers
        dataset_genes = self.process_gene_versions(dataset_genes)
        
        # Find common genes
        common_genes = set(dataset_genes).intersection(recon_genes)
        return common_genes

    def save_common_genes(self):
        """Save common genes to a specified output file."""
        common_genes = self.find_common_genes()
        with open(self.output_file, mode='w') as file:
            for gene in common_genes:
                file.write(gene + '\n')


In [ ]:
import pandas as pd

# # Example usage
# csv_file_path = 'transcriptomics.csv'  # Replace with your actual CSV file path
# gene_file_path = 'common_genes.txt'    # Replace with your actual text file path

# dataset_filter = DatasetFilter(csv_file_path, gene_file_path)
# dataset_filter.save_filtered_dataset()


class DatasetFilter:
    def __init__(self, csv_file, gene_file, output_file, metadata_columns):
        self.csv_file = csv_file
        self.gene_file = gene_file
        self.output_file = output_file
        self.metadata_columns = metadata_columns

    def load_dataset(self):
        """Load the dataset from a CSV file."""
        return pd.read_csv(self.csv_file)

    def load_required_genes(self):
        """Load the required genes from a text file."""
        with open(self.gene_file, 'r') as file:
            return [line.strip() for line in file]

    def filter_dataset(self):
        """Filter the dataset for required genes and metadata columns."""
        # Load the dataset and gene list
        df = self.load_dataset()
        required_genes = self.load_required_genes()

        # Remove gene version numbers from dataset column names
        df.columns = df.columns.str.split('.').str[0]

        # Combine metadata columns with the required gene columns
        columns_to_keep = self.metadata_columns + required_genes

        # Filter the dataframe to keep only the necessary columns
        filtered_df = df[columns_to_keep]
        return filtered_df

    def save_filtered_dataset(self):
        """Save the filtered dataset to a CSV file."""
        filtered_df = self.filter_dataset()
        filtered_df.to_csv(self.output_file, index=False)
        print(f"Filtered dataset saved as '{self.output_file}'")




In [ ]:
import pandas as pd
import json

# # Example usage
# filtered_dataset_path = 'filtered_dataset.csv'  # Replace with your actual filtered dataset file path
# gene_dict_path = 'gene_dict.json'               # Replace with your actual gene dictionary file path

# entrez_mapper = EntrezMapper(filtered_dataset_path, gene_dict_path)
# entrez_mapper.save_mapped_dataset()

class EntrezMapper:
    def __init__(self, filtered_dataset_file, gene_dict_file, output_file, metadata_columns):
        self.filtered_dataset_file = filtered_dataset_file
        self.gene_dict_file = gene_dict_file
        self.output_file = output_file
        self.metadata_columns = metadata_columns

    def load_data(self):
        """Load the filtered dataset and gene dictionary from files."""
        df = pd.read_csv(self.filtered_dataset_file)
        with open(self.gene_dict_file, 'r') as file:
            entrez_to_ensembl = json.load(file)
        return df, entrez_to_ensembl

    def map_entrez_ids(self):
        """Map Ensembl gene IDs to Entrez IDs and calculate maximum expression values."""
        df, entrez_to_ensembl = self.load_data()
        
        # Initialize a new DataFrame with metadata columns
        new_df = df[self.metadata_columns].copy()

        # Map each Entrez ID to the maximum expression of corresponding Ensembl IDs
        for entrez_id, ensembl_ids in entrez_to_ensembl.items():
            # Check for Ensembl IDs present in the dataset columns
            valid_ensembl_ids = [ensembl_id for ensembl_id in ensembl_ids if ensembl_id in df.columns]
            
            if valid_ensembl_ids:
                # If only one Ensembl ID is present, use its expression values directly
                if len(valid_ensembl_ids) == 1:
                    new_df[entrez_id] = df[valid_ensembl_ids[0]]
                else:
                    # Otherwise, calculate the max expression across the multiple Ensembl IDs
                    new_df[entrez_id] = df[valid_ensembl_ids].max(axis=1)
        return new_df

    def save_mapped_dataset(self):
        """Save the new dataset with Entrez IDs and calculated expression values."""
        new_df = self.map_entrez_ids()
        new_df.to_csv(self.output_file, index=False)
        print(f"Mapped dataset saved as '{self.output_file}'")


In [ ]:
import pandas as pd
from scipy.io import savemat

# # Example usage
# dataset_path = 'mapped_entrez_dataset.csv'  # Replace with your actual dataset file path

# mat_exporter = MATFileExporter(dataset_path)
# mat_exporter.save_to_mat()

class MATFileExporter:
    def __init__(self, dataset_file, output_file, metadata_columns):
        self.dataset_file = dataset_file
        self.output_file = output_file
        self.metadata_columns = metadata_columns

    def load_data(self):
        """Load the dataset from a CSV file."""
        return pd.read_csv(self.dataset_file)

    def process_data(self):
        """Process the dataset to calculate mean expression values by diagnosis."""
        df = self.load_data()

        # Identify gene columns by excluding metadata columns
        gene_columns = [col for col in df.columns if col not in self.metadata_columns]

        # Split data by diagnosis
        health_control_data = df[df['Factors'] == 'healthy'][gene_columns]
        
        # Alzheimer data is either 'AD' or 'AD+'
        alzheimer_data = df[df['Factors'].str.contains('c')][gene_columns]

        # Calculate mean expression values and reshape for MATLAB compatibility
        health_control_mean = health_control_data.mean(axis=0).values.reshape(-1, 1)
        alzheimer_mean = alzheimer_data.mean(axis=0).values.reshape(-1, 1)

        # Convert gene IDs from strings to floats and format as n x 1 lists for MATLAB
        gene_ids = [[float(col)] for col in gene_columns]

        # Return processed data
        return {
            'HealthControl': health_control_mean,
            'Alzheimer': alzheimer_mean,
            'GeneID': gene_ids
        }

    def save_to_mat(self):
        """Save the processed data to a MATLAB .mat file."""
        data_dict = self.process_data()
        savemat(self.output_file, data_dict)
        print(f"Data has been successfully saved to '{self.output_file}'")


In [ ]:
class Pipeline:
    def __init__(self, recon_json='Recon3D.json', entrez_ids_file='entrez_ids.txt', gene_dict_file='gene_dict.json',
                 csv_file='transcriptomics.csv', common_genes_txt_file='common_genes.txt',
                 mapped_entrez_dataset_file='mapped_entrez_dataset.csv',
                 filtered_dataset_file='filtered_dataset.csv', metadata_columns=['Factors'],
                 output_mat_file='mean_output_dataset.mat'):
        recon_json = 'data/' + recon_json
        csv_file = 'data/' + csv_file
        print('Initializing Pipeline...')
        self.gene_mapper = GeneMapper(recon_json, entrez_ids_file, common_genes_txt_file, gene_dict_file)
        print('GeneMapper initialized with files:', recon_json, entrez_ids_file, common_genes_txt_file, gene_dict_file)
        self.gene_processor = GeneProcessor(csv_file, common_genes_txt_file, common_genes_txt_file)
        print('GeneProcessor initialized with files:', csv_file, common_genes_txt_file)
        self.dataset_filter = DatasetFilter(csv_file, common_genes_txt_file, filtered_dataset_file, metadata_columns)
        print('DatasetFilter initialized with files:', csv_file, common_genes_txt_file, filtered_dataset_file)
        self.entrez_mapper = EntrezMapper(filtered_dataset_file=filtered_dataset_file, gene_dict_file=gene_dict_file, output_file=mapped_entrez_dataset_file, metadata_columns=metadata_columns)
        print('EntrezMapper initialized with files:', filtered_dataset_file, gene_dict_file, mapped_entrez_dataset_file)
        self.mat_exporter = MATFileExporter(dataset_file=mapped_entrez_dataset_file, output_file=output_mat_file, metadata_columns=metadata_columns)
        print('MATFileExporter initialized with files:', mapped_entrez_dataset_file, output_mat_file)

    def run(self):
        """Run the entire pipeline."""
        print('Starting Gene Mapping...')
        self.gene_mapper.run_mapping()
        print('Processing Genes...')
        self.gene_processor.save_common_genes()
        print('Filtering Dataset...')
        self.dataset_filter.save_filtered_dataset()
        print('Mapping Entrez IDs...')
        self.entrez_mapper.save_mapped_dataset()
        print('Exporting to MATLAB format...')
        self.mat_exporter.save_to_mat()
        print('Pipeline completed successfully!')


pipeline = Pipeline()
pipeline.run()
